# Euclid Lensed-AGN Forecast
Wide survey: 14500 deg$^2$, second-brightest-image cut at the VIS 5$\sigma$ depth (26.2 mag),
image separation 0.2''–4.0''. The Y/J/H magnitudes of every image are kept in the catalog, so
the NISP 5$\sigma$ depth (24.5 mag) is applied afterwards as a sub-selection.

Source redshifts are limited to $z_S < 3$ because the SkyPy/kcorrect host-galaxy templates are
not defined blueward of the Euclid filters above that redshift.

In [1]:
import os
import gzip
import pickle
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.cosmology import FlatLambdaCDM
from astropy.table import Table, vstack
from astropy.units import Quantity
from IPython.display import display

import slsim
import slsim.Pipelines as pipelines
from slsim.Deflectors.DeflectorPopulation.galaxy_deflectors import GalaxyDeflectors
from slsim.Lenses.LensPopulation.lens_pop import LensPop
from slsim.Sources.SourceCatalogues.QuasarCatalog.quasar_pop import QuasarRate
from slsim.Sources.SourcePopulation.point_plus_extended_sources import PointPlusExtendedSources
from slsim.Sources.SourceVariability.agn import agn_bounds_dict

from utils import extract_lensed_agn_properties, plot_euclid_montage, plot_survey_corner

warnings.filterwarnings("ignore")

%load_ext autoreload
%autoreload 2

        Use pytest instead. [astropy.tests.runner]
        Use pytest instead. [astropy.utils.decorators]


In [2]:
DATA_DIR = "./data"
FIGURE_DIR = "./figures"

EUCLID_BANDS = ["VIS", "Y", "J", "H"]
VIS_DEPTH = 26.2
NISP_DEPTH = 24.5

cosmo = FlatLambdaCDM(H0=70, Om0=0.3)
skypy_config = os.path.join(
    os.path.dirname(os.path.dirname(slsim.__file__)), "data/SkyPy/euclid-like.yml"
)
base_sky_area = Quantity(value=10, unit="deg2")
quasar_catalog_path = os.path.join(DATA_DIR, "euclid_quasar_source_catalog.fits")

## 1. Deflector and source populations

In [ ]:
galaxy_pipeline = pipelines.SkyPyPipeline(
    skypy_config=skypy_config, sky_area=base_sky_area, filters=None,
    cosmo=cosmo, z_min=0.0, z_max=3.0,
)
host_galaxy_candidates = vstack(
    [galaxy_pipeline.red_galaxies, galaxy_pipeline.blue_galaxies], join_type="exact"
)

lens_galaxies = GalaxyDeflectors(
    red_galaxy_list=galaxy_pipeline.red_galaxies,
    # blue_galaxy_list=galaxy_pipeline.blue_galaxies,
    kwargs_cut={"band": "VIS", "band_max": 28, "z_min": 0.01, "z_max": 2.5},
    kwargs_mass2light={}, cosmo=cosmo, sky_area=base_sky_area,
    gamma_pl={"mean": 2.078, "std_dev": 0.16},
)

In [4]:
if os.path.exists(quasar_catalog_path):
    quasar_source = Table.read(quasar_catalog_path)
else:
    quasar_source = QuasarRate(
        skypy_config=skypy_config, cosmo=cosmo, sky_area=base_sky_area, noise=True,
        redshifts=np.linspace(0.001, 3.01, 100),
        host_galaxy_candidate=host_galaxy_candidates,
        use_qsogen_sed=True, qsogen_bands=EUCLID_BANDS, use_sed_interpolator=True,
    ).quasar_sample(m_min=15, m_max=30, host_galaxy=True)
    quasar_source.write(quasar_catalog_path, overwrite=True)

rng = np.random.default_rng(42)
quasar_source["black_hole_spin"] = rng.uniform(
    *agn_bounds_dict["black_hole_spin_bounds"], len(quasar_source)
)
quasar_source["inclination_angle"] = rng.uniform(
    *agn_bounds_dict["inclination_angle_bounds"], len(quasar_source)
)

source_quasar = PointPlusExtendedSources(
    point_plus_extended_sources_list=quasar_source, cosmo=cosmo, sky_area=base_sky_area,
    kwargs_cut={"band": "VIS", "band_max": 28, "z_min": 0.001, "z_max": 5.0},
    catalog_type="skypy", point_source_type="quasar", extended_source_type="single_sersic",
)

In [ ]:
quasar_source

## 2. Draw the Euclid wide-survey population

In [5]:
survey_data = {
    "Euclid_Wide": {
        "name": "Euclid Wide",
        "sky_area": Quantity(value=14500, unit="deg2"),
        "kwargs_lens_cuts": {
            "min_image_separation": 0.2,
            "max_image_separation": 4.0,
            "second_brightest_image_cut": {"VIS": VIS_DEPTH},
        },
        "catalog_savepath": os.path.join(DATA_DIR, "euclid_wide_lensed_agn_catalog.fits"),
    },
}

In [14]:
data = survey_data["Euclid_Wide"]

lens_pop = LensPop(
        deflector_population=lens_galaxies, source_population=source_quasar,
        cosmo=cosmo, sky_area=data["sky_area"],
    )

selected_lenses = lens_pop.draw_population(
        speed_factor=10000, kwargs_lens_cuts=data["kwargs_lens_cuts"], verbose=True
    )

print(f"Number of lensed AGN in {data['name']}: {len(selected_lenses)}")

Drawing lens population:   0%|          | 0/94226 [00:00<?, ?it/s]

Drawing lens population:   0%|          | 30/94226 [01:27<76:21:55,  2.92s/it] 


KeyboardInterrupt: 

In [ ]:
# for data in survey_data.values():
#     lens_pop = LensPop(
#         deflector_population=lens_galaxies, source_population=source_quasar,
#         cosmo=cosmo, sky_area=data["sky_area"],
#     )
#     data["lens_objects"] = lens_pop.draw_population(
#         speed_factor=10000, kwargs_lens_cuts=data["kwargs_lens_cuts"], verbose=True
#     )
#     data["catalog"] = extract_lensed_agn_properties(
#         data["lens_objects"], all_bands=EUCLID_BANDS, max_num_images=5
#     )
#     print(f"{data['name']}: {len(data['catalog'])} lensed AGNs")

#     data["catalog"].write(data["catalog_savepath"], format="fits", overwrite=True)
#     with gzip.open(data["catalog_savepath"].replace(".fits", "_lens_objects.pkl.gz"), "wb") as f:
#         pickle.dump(data["lens_objects"], f)

## 3. NISP-detected sub-sample and summary

In [ ]:
wide = survey_data["Euclid_Wide"]
nisp_detected = np.all(
    [wide["catalog"][f"second_brightest_image_ps_mag_{b}"] < NISP_DEPTH for b in ["Y", "J", "H"]],
    axis=0,
)

survey_data["Euclid_Wide_NISP"] = {
    "name": "Euclid Wide (VIS+NISP)",
    "sky_area": wide["sky_area"],
    "kwargs_lens_cuts": {
        "min_image_separation": 0.2,
        "max_image_separation": 4.0,
        "second_brightest_image_cut": {"VIS": VIS_DEPTH, "Y": NISP_DEPTH, "J": NISP_DEPTH, "H": NISP_DEPTH},
    },
    "catalog": wide["catalog"][nisp_detected],
    "lens_objects": [l for l, keep in zip(wide["lens_objects"], nisp_detected) if keep],
}

In [ ]:
summary_rows = []
for data in survey_data.values():
    catalog = data["catalog"]
    n_quads = int(np.sum(catalog["num_images"] == 4))
    n_doubles = int(np.sum(catalog["num_images"] == 2))
    cuts = data["kwargs_lens_cuts"]
    summary_rows.append({
        "Survey": data["name"],
        "Sky Area (deg²)": f"{data['sky_area'].value:,.0f}",
        "Total Lensed-AGN": f"{len(catalog):,}",
        "Doubles": f"{n_doubles:,}",
        "Quads": f"{n_quads:,}",
        "Quad Fraction": f"{n_quads / max(len(catalog), 1):.1%}",
        "Image Sep Cuts (\")": f"{cuts['min_image_separation']} - {cuts['max_image_separation']}",
        "Mag Cuts (2nd Brightest Image)": ", ".join(
            f"{b} < {m}" for b, m in cuts["second_brightest_image_cut"].items()
        ),
    })

display(pd.DataFrame(summary_rows).set_index("Survey"))

## 4. Population properties

In [ ]:
plot_params = [
    "theta_E_arcsec", "max_time_delay_days", "deflector_velocity_dispersion",
    "deflector_light_R_eff_arcsec", "deflector_mag_VIS", "z_D", "z_S",
]
latex_labels = [
    r"$\theta_E$ (arcsec)", r"$\Delta t_{\max}$ (days)", r"$\sigma_{v, D}$ (km/s)",
    r"$R_{\rm eff, D}$ (arcsec)", r"$m_{D, \rm VIS}$", r"$z_D$", r"$z_S$",
]
range_vals = {
    "theta_E_arcsec": (0.1, 2.0),
    "max_time_delay_days": (0, 200),
    "deflector_velocity_dispersion": (80, 380),
    "deflector_light_R_eff_arcsec": (0, 2.5),
    "deflector_mag_VIS": (15, 25),
    "z_D": (0, 2.5),
    "z_S": (0, 3.1),
}

fig = plot_survey_corner(
    survey_data=survey_data,
    keys_to_plot=["Euclid_Wide", "Euclid_Wide_NISP"],
    params=plot_params, latex_labels=latex_labels, range_vals=range_vals,
    color_map={
        "Euclid_Wide": {"quads": "tab:blue", "doubles": "tab:cyan"},
        "Euclid_Wide_NISP": {"quads": "tab:red", "doubles": "tab:orange"},
    },
    title="Lensed AGN Forecast: Euclid Wide Survey",
    figsize=(16, 16), smooth=3, separate_quads_doubles=True,
    text_x=0.68, text_y_start=0.72, text_y_step=0.18,
    save_path=os.path.join(FIGURE_DIR, "corner_euclid_wide.png"),
)
plt.show()

## 5. Euclid RGB montage

In [ ]:
fig = plot_euclid_montage(
    lenses=wide["lens_objects"],
    number_to_plot=min(200, len(wide["lens_objects"])),
    num_cols=20, num_pix_vis=50,
    colour="VIS_WEIGHTED_Y_J_H", stretch="mtf",
    plot_title="Sample Lensed AGNs from Euclid Wide (VIS + Y/J/H)",
    random_seed=42,
)
if fig is not None:
    fig.savefig(os.path.join(FIGURE_DIR, "montage_Euclid_Wide.png"), dpi=300, bbox_inches="tight")
    plt.show()